# Modified Attention-PestNet with Lightweight HSDPA (L-HSDPA)
**Thesis Topic:** A Modified Attention-PestNet with Lightweight Hierarchical Scaled Dot-Product Attention for Efficient Insect Pest Detection  
**Authors:** DM Rashid P. Ferrer, James Oliver C. Mendoza, Lean Adrian B. Murillo (University of Mindanao)
---
This notebook sets up the environment, prepares the IP102 benchmark dataset, inspects model architecture (parameters & GFLOPs), and executes training on Kaggle GPU.

In [ ]:
# 1. Setup repository and install ultralytics
import os
import shutil

REPO_DIR = "/kaggle/working/HSDPA"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/90377Sednaaa/HSDPA.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
# Ensure L_HSDPA is in Blocks.py
with open("ultralytics/nn/ext/Blocks.py", "r") as f:
    blocks_code = f.read()

if "class L_HSDPA" not in blocks_code:
    l_hsdpa_def = """

class L_HSDPA(nn.Module):
    def __init__(self, c1, c2, r=0.5):
        super().__init__()
        self.r = float(r)
        c_mid = max(16, int(c1 * self.r))
        self.reduce_conv = nn.Conv2d(c1, c_mid, kernel_size=1, stride=1, bias=False)
        self.reduce_bn = nn.BatchNorm2d(c_mid)
        self.reduce_act = nn.SiLU()
        self.sdp = ScaleDotProduct(c_mid, c_mid)
        self.proj_conv = nn.Conv2d(c_mid * 4, c2, kernel_size=1, stride=1, bias=False)
        self.proj_bn = nn.BatchNorm2d(c2)
        self.proj_act = nn.SiLU()

    def forward(self, x):
        x_comp = self.reduce_act(self.reduce_bn(self.reduce_conv(x)))
        y1 = self.sdp(x_comp)
        y2 = self.sdp(y1)
        y3 = self.sdp(y2)
        cat = torch.cat((y1, y2, y3, x_comp), dim=1)
        out = self.proj_act(self.proj_bn(self.proj_conv(cat)))
        return out
"""
    with open("ultralytics/nn/ext/Blocks.py", "a") as f:
        f.write(l_hsdpa_def)
    print("Patched Blocks.py with L_HSDPA")

# Ensure L_HSDPA is in tasks.py
with open("ultralytics/nn/tasks.py", "r") as f:
    tasks_code = f.read()

if "L_HSDPA" not in tasks_code:
    tasks_code = tasks_code.replace(
        "ScaleDotProduct, Contigous_Att}:",
        "ScaleDotProduct, Contigous_Att, L_HSDPA}:"
    )
    with open("ultralytics/nn/tasks.py", "w") as f:
        f.write(tasks_code)
    print("Patched tasks.py with L_HSDPA")

# Ensure cfg/l_hsdpa.yaml exists
if not os.path.exists("cfg/l_hsdpa.yaml"):
    with open("cfg/hsdpa.yaml", "r") as f:
        hsdpa_cfg = f.read()
    lhsdpa_cfg = hsdpa_cfg.replace(
        "[6, 1, Contigous_Att, [512, 1024]]",
        "[6, 1, L_HSDPA, [512, 1024, 0.5]]"
    ).replace(
        "[4, 1, Contigous_Att, [256, 512]]",
        "[4, 1, L_HSDPA, [256, 512, 0.5]]"
    ).replace(
        "[9, 1, Contigous_Att, [1024, 2048]]",
        "[9, 1, L_HSDPA, [1024, 2048, 0.5]]"
    )
    with open("cfg/l_hsdpa.yaml", "w") as f:
        f.write(lhsdpa_cfg)
    print("Generated cfg/l_hsdpa.yaml")

!pip install -q -e .


In [ ]:
# 2. Check GPU environment and CUDA capabilities
import torch
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
device_count = torch.cuda.device_count()
print("GPU Count:", device_count)
for i in range(device_count):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


In [ ]:
# 3. Setup dataset configuration for IP102
import os
import yaml

DATA_ROOT = "/kaggle/input/ip102-yolov5/IP102_YOLOv5"
SOURCE_YAML = os.path.join(DATA_ROOT, "ip102.yaml")

with open(SOURCE_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

# Fix dataset paths for Kaggle read-only input mount
data_cfg["path"] = DATA_ROOT
data_cfg["train"] = "images/train"
data_cfg["val"] = "images/val"

FIXED_YAML = "/kaggle/working/ip102.yaml"
with open(FIXED_YAML, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f"Fixed dataset YAML written to: {FIXED_YAML}")
print(f"Number of classes configured: {len(data_cfg['names'])}")


In [ ]:
# 4. Compare Model Complexity: Baseline HSDPA vs. Proposed L-HSDPA
from ultralytics import YOLO

print("=" * 60)
print("BASELINE MODEL (Attention-PestNet / HSDPA):")
baseline_model = YOLO("cfg/hsdpa.yaml")
baseline_model.info(detailed=False)

print("=" * 60)
print("PROPOSED MODEL (L-HSDPA):")
lhsdpa_model = YOLO("cfg/l_hsdpa.yaml")
lhsdpa_model.info(detailed=False)
print("=" * 60)


In [ ]:
# 5. Train Model (L-HSDPA on IP102)
import torch
from ultralytics import YOLO

MODEL_CFG = "cfg/l_hsdpa.yaml"
PROJECT_NAME = "/kaggle/working/runs/L_HSDPA"
EXP_NAME = "ip102_lhsdpa"

model = YOLO(MODEL_CFG)

# Use multi-GPU if dual T4 is active, else single GPU
device_arg = [0, 1] if torch.cuda.device_count() >= 2 else 0

results = model.train(
    data="/kaggle/working/ip102.yaml",
    epochs=100,               # Stage 1: 100 epochs
    batch=16,                 # Batch size
    imgsz=640,
    device=device_arg,
    optimizer="SGD",          # Matching proposal SGD setup
    lr0=0.01,
    momentum=0.9,
    amp=True,                 # Mixed precision on T4
    save_period=10,           # Save every 10 epochs
    project=PROJECT_NAME,
    name=EXP_NAME,
    exist_ok=True,
    workers=2,
    verbose=True
)


In [ ]:
# 6. Verify and summarize output files
import os, glob

print("Generated weight checkpoints:")
for f in glob.glob(f"{PROJECT_NAME}/{EXP_NAME}/weights/*.pt"):
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  {f} ({size_mb:.2f} MB)")
